# Capítulo 11 · Laboratório 11: volume, o combustível do mercado
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap11.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. Os indicadores do capítulo escritos à mão, com pandas: OBV, WVAD, LAD, as duas versões do WAD, CMF, oscilador de Chaikin, MFI, Force Index, VPCI, MMPV, Intraday Intensity, VWAP e VWAP ancorada. O laboratório confere as contas do texto, mede a VWAP do dia e as VWAPs ancoradas (Brian Shannon), testa o toque contra a reação, as quatro linhas do VPCI (Buff Dormeier), o filtro do Force Index da 3ª edição, o rompimento com e sem volume e o aviso prévio do OBV. Atenção: o Yahoo guarda só cerca de 60 dias de barras de 5 minutos; fora dessa janela, a VWAP do dia da Parte 5 usa os últimos 30 dias, e o notebook avisa que os números diferem dos do livro.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap11.pkl"
pc = lambda x, k=1: f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=2: ("+" if x >= 0 else "") + pc(x, k)
nr = lambda x, k=2: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
ATIVOS = {"Ibovespa": "^BVSP", "BOVA11": "BOVA11.SA", "PETR4": "PETR4.SA", "VALE3": "VALE3.SA",
          "ITUB4": "ITUB4.SA", "WEGE3": "WEGE3.SA", "CMIG4": "CMIG4.SA"}
ACOES = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
         "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "BPAC11", "SBSP3", "CSAN3"]
CINCO = ["BOVA11", "PETR4", "VALE3", "ITUB4", "WEGE3"]


def baixa(t, ini, **kw):
    d = yf.download(t, start=ini, end=FIM, auto_adjust=True, progress=False, **kw)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    return d.dropna(subset=["Close"])


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


D = pd.read_pickle(CACHE) if os.path.exists(CACHE) else {}
for nome, t in ATIVOS.items():
    if nome not in D:
        D[nome] = baixa(t, "2000-01-01")
confere({nome: D[nome] for nome in ATIVOS})
for t in ACOES:
    if t not in D:
        D[t] = baixa(f"{t}.SA", "2010-01-01")
if "PETR4_5m" not in D:                 # o Yahoo só guarda cerca de 60 dias de barras de 5 minutos
    try:
        D["PETR4_5m"] = baixa("PETR4.SA", "2026-09-01", interval="5m")
    except Exception:
        D["PETR4_5m"] = pd.DataFrame(columns=["Open", "High", "Low", "Close", "Volume"])
    if D["PETR4_5m"].empty:             # fora da janela de 60 dias: usa os últimos 30 dias
        print("Aviso: o Yahoo só guarda 60 dias de barras de 5 minutos. A janela do livro (setembro de 2026) não está"
              " mais disponível; a Parte 5 usa os últimos 30 dias, e os números diferem dos do livro.")
        x = yf.download("PETR4.SA", period="30d", interval="5m", auto_adjust=True, progress=False)
        if isinstance(x.columns, pd.MultiIndex):
            x.columns = x.columns.get_level_values(0)
        if len(x):                      # sem nem isso, a Parte 5 roda só com a VWAP ancorada
            D["PETR4_5m"] = x.dropna(subset=["Close"])
pd.to_pickle(D, CACHE)

## Parte 1: dados

In [ ]:
def limpa(d):
    """Tira pregões sem volume e barras com preço impossível (a fonte gratuita traz alguns)."""
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] >= d["Low"]) & (d["Volume"] > 0)
    dentro = d[["Open", "Close"]].le(d["High"] * 1.001, axis=0).all(axis=1) & d[["Open", "Close"]].ge(d["Low"] * 0.999, axis=0).all(axis=1)
    return d[ok & dentro].copy()


A = {k: limpa(D[k]) for k in ATIVOS if k != "Ibovespa"}
A.update({t: limpa(D[t]) for t in ACOES if t not in A})
M5 = limpa(D["PETR4_5m"])
if len(M5) and M5.index.tz is not None:
    M5.index = M5.index.tz_convert("America/Sao_Paulo")
print(f"Parte 1. {len(A)} ativos com volume (BOVA11 desde {A['BOVA11'].index[0]:%Y}); último pregão"
      f" {A['PETR4'].index[-1]:%d/%m/%Y}; PETR4 em 5 minutos: {len(M5)} barras em"
      f" {len(set(M5.index.date)) if len(M5) else 0} pregões.")

## Parte 2: os indicadores, escritos à mão

In [ ]:
def mme(x, n):
    return x.ewm(span=n, adjust=False).mean()


def obv(d):
    sinal = np.sign(d["Close"].diff()).fillna(0)          # +1 alta, -1 baixa, 0 igual
    return (sinal * d["Volume"]).cumsum()


def wvad(d):
    fator = ((d["Close"] - d["Open"]) / (d["High"] - d["Low"])).replace([np.inf, -np.inf], np.nan).fillna(0)
    return (fator * d["Volume"]).cumsum(), fator             # o fator vai de -1 a +1


def multiplicador(d):
    """Chaikin: [(F - Mín) - (Máx - F)] / (Máx - Mín). Não usa a abertura."""
    m = ((d["Close"] - d["Low"]) - (d["High"] - d["Close"])) / (d["High"] - d["Low"])
    return m.replace([np.inf, -np.inf], np.nan).fillna(0)


def lad(d):
    return (multiplicador(d) * d["Volume"]).cumsum()


def wad(d, com_volume=False):
    """A/D de Williams com a máxima e a mínima verdadeiras. Achelis (MetaStock, TA-Lib) não usa volume."""
    c0 = d["Close"].shift()
    trh, trl = np.maximum(d["High"], c0), np.minimum(d["Low"], c0)
    mov = np.where(d["Close"] > c0, d["Close"] - trl, np.where(d["Close"] < c0, d["Close"] - trh, 0))
    mov = pd.Series(mov, d.index).fillna(0)
    return (mov * d["Volume"] if com_volume else mov).cumsum()


def cmf(d, n=21):
    return (multiplicador(d) * d["Volume"]).rolling(n).sum() / d["Volume"].rolling(n).sum()


def chaikin_osc(d):
    x = lad(d)
    return mme(x, 3) - mme(x, 10)                            # diferença, e não razão


def mfi(d, n=14):
    pt = (d["High"] + d["Low"] + d["Close"]) / 3
    fluxo = pt * d["Volume"]
    pos = fluxo.where(pt > pt.shift(), 0).rolling(n).sum()
    neg = fluxo.where(pt < pt.shift(), 0).rolling(n).sum()
    return 100 - 100 / (1 + pos / neg)


def force(d, n=13):
    return mme(d["Close"].diff() * d["Volume"], n)


def vwma(c, v, n):
    return (c * v).rolling(n).sum() / v.rolling(n).sum()


def vpci(d, curto=5, longo=20):
    """VPCI = VPC × VPR × VM (Dormeier, S&C, jul. 2007). Os períodos variam entre as fontes."""
    c, v = d["Close"], d["Volume"]
    vpc = vwma(c, v, longo) - c.rolling(longo).mean()
    vpr = vwma(c, v, curto) / c.rolling(curto).mean()
    vm = v.rolling(curto).mean() / v.rolling(longo).mean()
    return vpc * vpr * vm


def intraday_intensity(d):
    """David Bostian: [(2F - Máx - Mín) / (Máx - Mín)] × V."""
    m = ((2 * d["Close"] - d["High"] - d["Low"]) / (d["High"] - d["Low"])).replace([np.inf, -np.inf], np.nan).fillna(0)
    return m * d["Volume"]


def preco_tipico(d):
    return (d["High"] + d["Low"] + d["Close"]) / 3


def vwap_ancorada(d, ancora):
    x = d.loc[ancora:]
    return (preco_tipico(x) * x["Volume"]).cumsum() / x["Volume"].cumsum()


p = A["PETR4"]
ult = p.index[-1]
print(f"\nParte 2. PETR4 em {ult:%d/%m/%Y}: OBV {nr(obv(p).iloc[-1] / 1e6, 0)} milhões | CMF(21) {nr(cmf(p).iloc[-1], 3)}"
      f" | MFI(14) {nr(mfi(p).iloc[-1], 1)} | Force(13) {nr(force(p).iloc[-1] / 1e6, 1)} milhões |"
      f" VPCI {nr(vpci(p).iloc[-1], 4)}")

## Parte 3: as contas do texto

In [ ]:
mmpv = (20 * 100_000 + 22 * 300_000) / 400_000
_, fat = wvad(p)
iguais = np.allclose(intraday_intensity(p), multiplicador(p) * p["Volume"])
print(f"\nParte 3. MMPV do exemplo: {nr(mmpv, 1)} (a MMS dá 21,0). Fator do WVAD na PETR4: de {nr(fat.min())} a"
      f" {nr(fat.max())} (vai de -1 a +1, e não de 0% a 100%).")
print(f"   Intraday Intensity de Bostian igual ao volume de fluxo de Chaikin em todos os dias? {'sim' if iguais else 'não'}")
w0, w1 = wad(p), wad(p, True)
print(f"   WAD sem volume x com volume (PETR4, desde 2000): correlação das variações diárias"
      f" {nr(w0.diff().corr(w1.diff()), 2)}")

## Parte 4: osciladores limitados?

In [ ]:
FAIXA = {}
print("\nParte 4. Mínimo e máximo de cada oscilador desde 2010")
for nome in ("PETR4", "VALE3"):
    d = A[nome].loc["2010":]
    FAIXA[nome] = {"MFI(14)": (mfi(d).min(), mfi(d).max()), "CMF(21)": (cmf(d).min(), cmf(d).max()),
                   "Oscilador Chaikin": (chaikin_osc(d).min(), chaikin_osc(d).max()),
                   "Force Index(13)": (force(d).min(), force(d).max()), "VPCI": (vpci(d).min(), vpci(d).max())}
    for k, (a, b) in FAIXA[nome].items():
        esc = 1e6 if abs(b) > 1e5 else 1
        print(f"   {nome:<6} {k:<18} de {nr(a / esc)} a {nr(b / esc)}{' milhões' if esc > 1 else ''}")

## Parte 5: VWAP do dia e VWAP ancorada

In [ ]:
VWAP_DIA = {}
if len(M5):
    dia = M5.index.date
    vw = (preco_tipico(M5) * M5["Volume"]).groupby(dia).cumsum() / M5["Volume"].groupby(dia).cumsum()
    fech = M5.groupby(dia).tail(1)
    acima = fech["Close"] > vw.loc[fech.index]
    VWAP_DIA = {"barras": M5, "vwap": vw, "dias": len(fech), "acima": acima.mean()}
    quando = ("de setembro de 2026" if f"{M5.index[0]:%Y-%m}" == "2026-09"
              else f"de {M5.index[0]:%d/%m/%Y} a {M5.index[-1]:%d/%m/%Y}")
    print(f"\nParte 5. VWAP do dia na PETR4 (5 minutos, {len(fech)} pregões {quando}): fechou acima da VWAP em"
          f" {pc(acima.mean(), 0)} dos pregões; último: fechamento R$ {nr(fech['Close'].iloc[-1])}, VWAP"
          f" R$ {nr(vw.iloc[-1])}")
AVWAP = {}
print("   VWAP ancorada (preço médio de quem comprou desde a âncora) em 30/09/2026")
for nome in CINCO + ["CMIG4"]:
    d = A[nome]
    ano = d.iloc[-252:]
    ancoras = {"fundo de 12 meses": ano["Low"].idxmin(), "topo de 12 meses": ano["High"].idxmax(),
               "início do ano": d.loc[str(d.index[-1].year)].index[0]}
    AVWAP[nome] = {}
    for k, dt in ancoras.items():
        av, c = vwap_ancorada(d, dt).iloc[-1], d["Close"].iloc[-1]
        AVWAP[nome][k] = (dt, av, c / av - 1)
    print(f"   {nome:<6} preço R$ {nr(d['Close'].iloc[-1])} | " + " | ".join(
        f"{k} ({v[0]:%d/%m/%y}): {ps(v[2], 1)}" for k, v in AVWAP[nome].items()))

In [ ]:
import matplotlib.pyplot as plt
d = A["VALE3"].iloc[-290:]
plt.figure(figsize=(9, 4))
plt.plot(d.index, d["Close"], color="black", lw=0.8, label="VALE3")
for k, (dt, av, dist) in AVWAP["VALE3"].items():
    plt.plot(vwap_ancorada(A["VALE3"], dt), lw=1.2, label=f"VWAP ancorada: {k}")
plt.legend()
plt.title("Quem comprou desde cada âncora está ganhando ou perdendo?")
plt.show()

## Parte 6: tocar a VWAP ancorada ou esperar a reação?

In [ ]:
def toque_e_reacao(d, folga=0.02, janela=5, h=20):
    """VWAP ancorada no 1º pregão de cada ano. Toque: a mínima encosta na AVWAP vindo de cima e o fechamento fica acima.
    Reação: primeiro fechamento 2% acima da AVWAP até 5 pregões depois de um toque."""
    toques, reacoes, acima = [], [], []
    f = d["Close"].shift(-h) / d["Close"] - 1
    for ano in sorted(set(d.index.year)):
        x = d.loc[str(ano)]
        if len(x) < 40:
            continue
        av = vwap_ancorada(x, x.index[0])
        t = (x["Low"] <= av) & (x["Close"] >= av) & (x["Close"].shift() > av.shift())
        t.iloc[:20] = False                                  # a AVWAP precisa de alguns pregões para valer
        ult_toque = pd.Series(np.where(t, np.arange(len(x)), np.nan), x.index).ffill()
        dist = np.arange(len(x)) - ult_toque
        r = (x["Close"] > av * (1 + folga)) & (dist > 0) & (dist <= janela)
        r &= ~r.shift(fill_value=False)
        toques += list(f.reindex(x.index[t]).dropna())
        reacoes += list(f.reindex(x.index[r]).dropna())
        acima += list(f.reindex(x.index[(x["Close"] > av)][20:]).dropna())
    return np.array(toques), np.array(reacoes), np.array(acima)


TOQUE = {}
print("\nParte 6. VWAP ancorada no início de cada ano (2010 a 2026): retorno médio dos 20 pregões seguintes")
for nome in CINCO:
    t, r, a = toque_e_reacao(A[nome].loc["2010":])
    TOQUE[nome] = (len(t), t.mean(), (t > 0).mean(), len(r), r.mean(), (r > 0).mean(), a.mean())
    print(f"   {nome:<6} toque: {len(t):>3} casos, {ps(t.mean())} (positivos {pc((t > 0).mean(), 0)}) | reação: {len(r):>3}"
          f" casos, {ps(r.mean())} (positivos {pc((r > 0).mean(), 0)}) | qualquer dia acima: {ps(a.mean())}")

## Parte 7: VPCI e a Tabela 11.3

In [ ]:
VP = {}
print("\nParte 7. VPCI (5 e 20): tendência do preço em 20 pregões x VPCI subindo ou caindo em 5 pregões (2010 a 2026)")
print("   Retorno médio dos 20 pregões seguintes")
for nome in CINCO:
    d = A[nome].loc["2010":]
    c, x = d["Close"], vpci(d)
    sobe, vsobe = c > c.shift(20), x > x.shift(5)
    f = c.shift(-20) / c - 1
    VP[nome] = {(a, b): f[(sobe == a) & (vsobe == b) & x.notna()].dropna() for a in (True, False) for b in (True, False)}
    print(f"   {nome:<6} " + " | ".join(f"preço {'sobe' if a else 'cai'}, VPCI {'sobe' if b else 'cai'}: {ps(s.mean())}"
                                       for (a, b), s in VP[nome].items()))

## Parte 8: Force Index e o filtro da 3ª edição

In [ ]:
FI = {}
print("\nParte 8. Force Index: compra com MME de 22 subindo e Force Index de 2 abaixo de zero (2010 a 2026)")
print("   Retorno médio dos 10 pregões seguintes, sem e com o filtro 'preço até 1% da MME de 22'")
for nome in CINCO:
    d = A[nome].loc["2010":]
    c = d["Close"]
    fi2, m22 = force(d, 2), mme(c, 22)
    base = (m22 > m22.shift()) & (fi2 < 0)
    perto = base & ((c / m22 - 1).abs() <= 0.01)
    f = c.shift(-10) / c - 1
    a, b, t = f[base].dropna(), f[perto].dropna(), f.dropna()
    FI[nome] = (len(a), a.mean(), len(b), b.mean(), t.mean())
    print(f"   {nome:<6} regra: {ps(a.mean())} ({len(a)}) | com filtro: {ps(b.mean())} ({len(b)}) | qualquer dia:"
          f" {ps(t.mean())}")

## Parte 9: rompimento e volume relativo

In [ ]:
linhas = []
for t in ACOES:
    d = A[t].loc["2012":]
    teto = d["High"].rolling(20).max().shift()
    rv = d["Volume"] / d["Volume"].rolling(20).mean().shift()
    rompe = (d["Close"] > teto) & (d["Close"].shift() <= teto.shift())
    volta = pd.concat([d["Close"].shift(-k) < teto for k in (1, 2, 3)], axis=1).any(axis=1)
    f = d["Close"].shift(-20) / d["Close"] - 1
    linhas.append(pd.DataFrame({"rv": rv[rompe], "volta": volta[rompe], "f20": f[rompe]}).dropna())
ROMP = pd.concat(linhas)
ROMP["faixa"] = pd.cut(ROMP["rv"], [0, 1, 2, np.inf], labels=["abaixo de 1x", "de 1x a 2x", "acima de 2x"])
RV = ROMP.groupby("faixa", observed=True).agg(casos=("rv", "size"), volta=("volta", "mean"), f20=("f20", "mean"))
print(f"\nParte 9. {len(ROMP)} rompimentos da máxima de 20 pregões em 20 ações (2012 a 2026), pelo volume relativo do dia")
for k, r in RV.iterrows():
    print(f"   volume {k:<13} {int(r['casos']):>5} casos | voltou para dentro em 3 pregões: {pc(r['volta'], 0)} |"
          f" retorno médio em 20 pregões: {ps(r['f20'])}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3.5))
plt.bar([str(k) for k in RV.index], RV["volta"] * 100, color=["lightgray", "gray", "black"])
plt.ylabel("voltou para dentro em 3 pregões (%)")
plt.title("Rompimentos da máxima de 20 pregões pelo volume relativo")
plt.show()

## Parte 10: o OBV rompe antes do preço?

In [ ]:
res = []
for t in ACOES:
    d = A[t].loc["2012":]
    c, o = d["Close"], obv(d)
    preco_novo = c >= c.rolling(20).max()
    obv_novo = (o >= o.rolling(20).max()) & ~preco_novo & (c >= c.rolling(20).max() * 0.97)
    sem = ~preco_novo & ~(o >= o.rolling(20).max()) & (c >= c.rolling(20).max() * 0.97)
    depois = pd.concat([preco_novo.shift(-k) for k in range(1, 11)], axis=1).fillna(False).any(axis=1)
    res.append((depois[obv_novo].sum(), obv_novo.sum(), depois[sem].sum(), sem.sum()))
r = np.array(res).sum(axis=0)
OBV_T = (r[0] / r[1], int(r[1]), r[2] / r[3], int(r[3]))
print(f"\nParte 10. Preço até 3% abaixo da máxima de 20 pregões, em 20 ações (2012 a 2026):")
print(f"   com o OBV em máxima de 20 pregões: o preço rompeu nos 10 pregões seguintes em {pc(OBV_T[0], 0)} dos"
      f" {OBV_T[1]} casos | sem o OBV na máxima: {pc(OBV_T[2], 0)} dos {OBV_T[3]} casos")

## Parte 11: cuidados com os dados

In [ ]:
ib = D["Ibovespa"]
print("\nParte 11. Dados de volume na fonte gratuita")
print(f"   Ibovespa: {pc((ib['Volume'] == 0).mean(), 0)} dos pregões com volume zero desde 2000 (índice não tem volume"
      f" próprio; use o BOVA11 ou o futuro)")
for nome in ("BOVA11", "PETR4", "CMIG4"):
    x = D[nome]
    print(f"   {nome:<6} pregões com volume zero: {(x['Volume'] == 0).sum()} | com máxima igual à mínima:"
          f" {(x['High'] == x['Low']).sum()} | descartados pela limpeza: {len(x) - len(limpa(x))} de {len(x)}")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, as funções `multiplicador`, `lad` e `intraday_intensity` da Parte 2. Mostre por que o multiplicador vale zero quando o fechamento fica no meio da barra e por que a conta de Bostian dá o mesmo número que o volume vezes o multiplicador de Chaikin." Confira com a Parte 3, que imprime se o Intraday Intensity de Bostian é igual ao volume de fluxo de Chaikin em todos os dias.
2. **Adaptar:** "Acrescente [ATIVO] ao dicionário `ATIVOS` da célula de abertura, com o ticker do Yahoo Finance, e inclua o nome dele na lista da Parte 5 (`CINCO + ["CMIG4"]`). Use uma ação ou um ETF, porque índice não tem volume próprio. Não mude a fórmula da VWAP ancorada nem as âncoras." Antes de olhar o ativo novo, confira se a Parte 5 continua mostrando a VALE3 a R$ 69,91, 5,2% abaixo da VWAP ancorada no fundo de 12 meses, e a PETR4 27,1% acima da dela, como no livro.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 6, ancore a VWAP no topo de cada ano em vez do primeiro pregão. O toque fica melhor ou pior?
2. Na Parte 7, troque os períodos do VPCI para 7 e 28, os do artigo de Dormeier. A Tabela 11.3 muda?
3. Na Parte 9, troque a máxima de 20 pregões pela de 55. O volume continua separando os rompimentos?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.